# Phase 2 — CoAtNet-2.5D Training (RSNA Knee Abnormality Detection)

**Version:** phase2-train-v1 | **Date:** 2026-10-05

Trains a 12-head multi-label classifier on Phase 0 preprocessed volumes + Phase 1 weak labels.
Metric: macro AUC. Backbone: CoAtNet (timm) with per-finding attention MIL over slices.

**Pipeline:** PROBE mode (tiny end-to-end validation) → Stage 1 (weak labels) → Stage 2 (gold fine-tune).
Set `PROBE = True` first. Only set `PROBE = False` after the probe prints ALL GREEN.

**Required Kaggle datasets attached:** `phase0runA`, `phase0runB`, `phase0runC`,
`rsna-knee-phase1-weak-labels-v1` (contains `weak_labels_v1.csv`).
**GPU:** 2x T4. **Never** P100.


In [ ]:
# ================= CONFIG (every knob lives here) =================
VERSION = "phase2-train-v12 (2026-10-05: single-GPU, no DataParallel)"
print(f"VERSION: {VERSION}")
PROBE = True              # True: 16 studies, 1 epoch, validate pipeline. False: full run.
PROBE_N_STUDIES = 16

# --- data ---
# 224 = coatnet native. 256 breaks its fixed attention (196 tokens expected).
IMAGE_SIZE = 224          # backbone input (cache is 336)
# 8 slices (not 16): CoAtNet OOMs on T4 with 16 (96 images/batch).
# 8 still covers the volume; 16 is a luxury we can't afford on 14.5GB.
N_SLICES = 8             # uniform slices sampled per plane
PLANES = ['axial', 'coronal', 'sagittal']

# --- model ---
# Prefer _rw_224: Ross Wightman trained, HAVE pretrained weights.
# Plain coatnet_0_224 has NONE in this timm version -> silent random init.
BACKBONE_CANDIDATES = ['coatnet_0_rw_224', 'coatnet_1_rw_224', 'coatnet_0_224', 'coatnet_0']  # tried in order; first hit wins
PRETRAINED = True         # ImageNet init (falls back to random with warning)
ATTN_DIM = 128
N_FINDINGS = 12

# --- training ---
# Single-GPU: 2 samples x 24 images = 48/batch (probe: 13.76 GB peak, fits).
# 4 would OOM (96 images). Effective batch via ACCUM_STEPS stays 16.
BATCH_SIZE = 2            # total across GPUs (2/GPU on 2xT4)
ACCUM_STEPS = 8          # 2x8=16 effective (was 4x4=16)           # effective batch = 16
EPOCHS_S1 = 8
EPOCHS_S2 = 5
LR_S1 = 1e-4
LR_S2 = 1e-5
WEIGHT_DECAY = 1e-4
GOLD_UPWEIGHT = 8.0

# --- ASL loss ---
ASL_GAMMA_POS = 0.0
ASL_GAMMA_NEG = 4.0
ASL_MARGIN = 0.05

# --- folds ---
N_FOLDS = 5
VAL_FOLD = 0
SEED = 0

# --- io ---
CKPT_DIR = '/kaggle/working/phase2_ckpt'
RESUME = None             # set to ckpt path to resume, e.g. f'{CKPT_DIR}/stage1_ep3.pt'

LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

print('CONFIG DUMP')
for k in ['PROBE','IMAGE_SIZE','N_SLICES','BACKBONE_CANDIDATES','PRETRAINED','BATCH_SIZE',
          'ACCUM_STEPS','EPOCHS_S1','EPOCHS_S2','LR_S1','LR_S2','GOLD_UPWEIGHT',
          'ASL_GAMMA_POS','ASL_GAMMA_NEG','ASL_MARGIN','N_FOLDS','VAL_FOLD','SEED','CKPT_DIR']:
    print(f'  {k} = {globals()[k]}')


In [ ]:
# ================= SETUP: imports, device, seeds, attachments =================
import os, sys, json, time, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f'torch {torch.__version__} | cuda: {torch.cuda.is_available()}',
      f' | gpus: {torch.cuda.device_count()}')
assert torch.cuda.is_available(), 'FATAL: no GPU'
NGPU = torch.cuda.device_count()
for i in range(NGPU):
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)}')
    assert 'T4' in torch.cuda.get_device_name(i), 'FATAL: need T4, not P100'
DEVICE = torch.device('cuda:0')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# ---- attachment check: find datasets at depth >= 4 ----
def find_under(root, name, max_depth=6):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep)
        if depth - root.count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if name in filenames or name in dirnames:
            hits.append(os.path.join(dirpath, name))
    return hits

print('\nattachment scan (/kaggle/input):')
for req in ['manifest.json', 'weak_labels_v1.csv']:
    hits = find_under('/kaggle/input', req)
    print(f'  {req}: {len(hits)} hit(s)')
    for h in hits[:6]:
        print(f'    {h}')
    assert hits, f'FATAL: {req} not found under /kaggle/input — attach the dataset'

MANIFEST_PATHS = find_under('/kaggle/input', 'manifest.json')
WEAK_CSV = find_under('/kaggle/input', 'weak_labels_v1.csv')[0]
TRAIN_CSV_CANDS = find_under('/kaggle/input', 'train.csv')
TRAIN_CSV = TRAIN_CSV_CANDS[0] if TRAIN_CSV_CANDS else None
print(f'\nmanifests: {len(MANIFEST_PATHS)} | weak csv: {WEAK_CSV}')
print(f'train.csv: {TRAIN_CSV} (for gold labels + priors)')
os.makedirs(CKPT_DIR, exist_ok=True)


In [ ]:
# ================= CORE MATH (tested locally; do not modify without re-testing) =================
# Asymmetric Loss (Ridnik et al. 2021), implemented from scratch.
def asl_loss(logits, targets, gamma_pos=ASL_GAMMA_POS, gamma_neg=ASL_GAMMA_NEG,
             margin=ASL_MARGIN, eps=1e-8):
    """logits, targets: (B, 12). Soft targets supported. Returns (B, 12) element losses."""
    p = torch.sigmoid(logits).clamp(eps, 1 - eps)
    loss_pos = -((1 - p) ** gamma_pos) * torch.log(p)
    p_m = (p - margin).clamp(min=0).clamp(eps, 1 - eps)
    loss_neg = -((p_m) ** gamma_neg) * torch.log(1 - p_m)
    return targets * loss_pos + (1 - targets) * loss_neg


# Mann-Whitney rank AUC from scratch (no sklearn). Ties get average rank.
def auc_score(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    n1, n0 = len(pos), len(neg)
    labs = np.concatenate([np.ones(n1), np.zeros(n0)])
    sc = np.concatenate([pos, neg])
    order = np.argsort(sc, kind='mergesort')
    s_sorted = sc[order]
    ranks = np.empty(len(sc))
    i = 0
    while i < len(sc):
        j = i
        while j < len(sc) and s_sorted[j] == s_sorted[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    r1 = ranks[labs == 1].sum()
    return (r1 - n1 * (n1 + 1) / 2.0) / (n1 * n0)


def macro_auc(y_true_2d, y_score_2d, labels=LABELS):
    aucs = {}
    for j, lab in enumerate(labels):
        a = auc_score(y_true_2d[:, j], y_score_2d[:, j])
        aucs[lab] = a
    valid = [a for a in aucs.values() if a is not None]
    return (sum(valid) / len(valid) if valid else None), aucs


# ---- scanner-grouped folds (proxy: StudyInstanceUID prefix) ----
# The Phase 0 manifest carries no scanner ID, so we group by UID prefix.
# DICOM UIDs embed the institution/scanner root as a prefix; studies sharing
# a prefix almost surely come from the same scanner. This is a documented
# PROXY, not a measured scanner fingerprint. Random K-fold inflates macro
# AUC by 0.087 (measured) and is forbidden by the execution protocol.
def scanner_group_key(study_uid, n_parts=8):
    return '.'.join(study_uid.split('.')[:n_parts])


def assign_folds(study_uids, n_folds=N_FOLDS, seed=SEED):
    rng = np.random.RandomState(seed)
    groups = {}
    for uid in study_uids:
        groups.setdefault(scanner_group_key(uid), []).append(uid)
    keys = sorted(groups.keys())
    rng.shuffle(keys)
    fold_of = {}
    for i, k in enumerate(keys):
        for uid in groups[k]:
            fold_of[uid] = i % n_folds
    return fold_of, groups


# self-test (runs in <1s, must print ALL GREEN)
_lt = torch.tensor([[0.0, 2.0, -2.0]]); _tg = torch.tensor([[1.0, 1.0, 0.0]])
_L = asl_loss(_lt, _tg)
assert _L[0, 1] < _L[0, 0] and _L[0, 2] < _L[0, 0], 'ASL focusing broken'
assert abs(auc_score([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]) - 1.0) < 1e-9
assert abs(auc_score([0, 1], [0.5, 0.5]) - 0.5) < 1e-9
assert auc_score([1, 1], [0.5, 0.6]) is None
print('core math self-test: ALL GREEN (ASL focusing, AUC ties/undefined)')


In [ ]:
# ================= DATA LOADING: cache index =================
# Shard layout (Phase 0): {cache}/shard_XXXX.npz with keys '{study_uid}/{plane}'
# -> uint8 (S, 336, 336). manifest.json: studies -> {shard, planes: {plane: {...}}}.
class ShardStore:
    def __init__(self, manifest_paths):
        self.studies = {}   # uid -> (cache_dir, manifest_entry)
        self._z = {}        # (cache_dir, shard) -> npz handle
        for mp in manifest_paths:
            cache_dir = str(Path(mp).parent)
            man = json.load(open(mp))
            for uid, entry in man['studies'].items():
                if uid in self.studies:
                    raise RuntimeError(f'duplicate study {uid} across caches')
                self.studies[uid] = (cache_dir, entry)
        print(f'indexed {len(self.studies)} studies from {len(manifest_paths)} manifests')

    def get_plane(self, uid, plane):
        # pid-aware cache: fork-safe when DataLoader uses multiple workers.
        # Each worker process opens its own handles on first access.
        pid = os.getpid()
        cache_dir, entry = self.studies[uid]
        shard = entry['shard']
        key = (pid, cache_dir, shard)
        z = self._z.get(key)
        if z is None:
            for k in [k for k in self._z if k[0] != pid]:
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
            z = np.load(os.path.join(cache_dir, shard))
            self._z[key] = z
            while len(self._z) > 4:
                k = next(iter(self._z))
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
        return z[f'{uid}/{plane}']

    def planes_of(self, uid):
        return list(self.studies[uid][1]['planes'].keys())


store = ShardStore(MANIFEST_PATHS)
# sanity: plane counts
from collections import Counter
pc = Counter()
for uid in store.studies:
    pc[tuple(sorted(store.planes_of(uid)))] += 1
print('plane-set distribution:')
for k, v in pc.most_common():
    print(f'  {k}: {v}')
n_missing = sum(1 for uid in store.studies if len(store.planes_of(uid)) < 3)
print(f'studies with <3 planes: {n_missing} (must be tolerated, never dropped)')


In [ ]:
# ================= LABELS: weak soft targets + gold override + abstain mask =================
weak = pd.read_csv(WEAK_CSV, encoding='latin-1')
print(f'weak_labels: {weak.shape} | cols ok: {list(weak.columns) == ["StudyInstanceUID"] + LABELS}')
assert list(weak.columns) == ['StudyInstanceUID'] + LABELS
assert weak[LABELS].notna().all().all(), 'FATAL: NaN in weak labels csv'
assert ((weak[LABELS] >= 0) & (weak[LABELS] <= 1)).all().all(), 'FATAL: labels outside [0,1]'

# gold hard labels + priors from train.csv (competition input)
gold = {}
priors = None
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
    tr = pd.read_csv(TRAIN_CSV, encoding='latin-1', usecols=['StudyInstanceUID'] + LABELS)
    gm = tr[LABELS].notna().all(axis=1)
    gdf = tr[gm]
    print(f'gold studies in train.csv: {len(gdf)}')
    for _, r in gdf.iterrows():
        gold[r['StudyInstanceUID']] = r[LABELS].values.astype(np.float32)
    priors = gdf[LABELS].mean().values.astype(np.float32)
else:
    print('WARNING: train.csv not found — gold override disabled, priors from weak means')
    priors = weak[LABELS].mean().values.astype(np.float32)
print('priors:', dict(zip(LABELS, np.round(priors, 3))))

# build per-study (targets[12], loss_mask[12], sample_weight, is_gold)
# abstain rule: |weak - prior| < 1e-6  =>  report silent  =>  mask loss (no gradient).
# (Phase 1 wrote the gold-mean prior exactly for 'not_mentioned'; real rank labels
#  are discrete fractions k/(n_terms-1) and cannot collide with the prior.)
weak_idx = weak.set_index('StudyInstanceUID')
EPS_ABSTAIN = 1e-6
label_pack = {}
n_gold_hit = n_abstain = 0
for uid in store.studies:
    if uid in gold:
        label_pack[uid] = (gold[uid], np.ones(12, np.float32), GOLD_UPWEIGHT, True)
        n_gold_hit += 1
    else:
        w = weak_idx.loc[uid, LABELS].values.astype(np.float32)
        mask = (np.abs(w - priors) > EPS_ABSTAIN).astype(np.float32)
        n_abstain += int((mask == 0).sum())
        label_pack[uid] = (w, mask, 1.0, False)
print(f'label packs: {len(label_pack)} | gold override: {n_gold_hit} | '
      f'abstained head-labels: {n_abstain} '
      f'({n_abstain / max(1, 12 * (len(label_pack) - n_gold_hit)):.1%} of weak)')
assert n_gold_hit == len(gold), 'some gold studies missing from cache!'


In [ ]:
# ================= FOLDS: scanner-grouped (proxy), never random =================
all_uids = sorted(store.studies.keys())
fold_of, groups = assign_folds(all_uids)
# Fallback: UID-prefix proxy collapsed (1 group) -> seeded random folds.
# Scanner grouping is preferred (random inflates AUC ~0.087 for *evaluation*),
# but an empty train set crashes. This keeps train/val non-empty and
# deterministic. Loud warning so nobody mistakes it for scanner-grouped.
import random as _r
if len(groups) < 2:
    print(f'WARNING: only {len(groups)} scanner-proxy group(s); FALLBACK to seeded random folds (not scanner-grouped!)')
    _rng = _r.Random(SEED)
    _shuf = sorted(all_uids)
    _rng.shuffle(_shuf)
    fold_of = {uid: (idx % N_FOLDS) for idx, uid in enumerate(_shuf)}
    groups = {f'random-fold-{f}': [u for u in all_uids if fold_of[u] == f] for f in range(N_FOLDS)}
print(f'{len(groups)} groups -> {N_FOLDS} folds')
for f in range(N_FOLDS):
    members = [u for u in all_uids if fold_of[u] == f]
    g_gold = sum(1 for u in members if label_pack[u][3])
    print(f'  fold {f}: {len(members)} studies, {g_gold} gold')
train_uids = [u for u in all_uids if fold_of[u] != VAL_FOLD]
val_uids = [u for u in all_uids if fold_of[u] == VAL_FOLD]
print(f'train: {len(train_uids)} | val: {len(val_uids)} (fold {VAL_FOLD})')
if PROBE:
    train_uids = train_uids[:PROBE_N_STUDIES]
    val_uids = val_uids[:max(4, PROBE_N_STUDIES // 4)]
    print(f'PROBE subset -> train: {len(train_uids)} | val: {len(val_uids)}')


In [ ]:
# ================= DATASET (no horizontal flips — laterality unknown) =================
def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n:
        return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]


class KneeDataset(Dataset):
    def __init__(self, uids, store, label_pack, image_size=IMAGE_SIZE,
                 n_slices=N_SLICES, augment=False):
        self.uids = uids
        self.store = store
        self.label_pack = label_pack
        self.image_size = image_size
        self.n_slices = n_slices
        self.augment = augment
        self.rng = np.random.RandomState(SEED)

    def __len__(self):
        return len(self.uids)

    def _augment(self, img):
        # img: float32 (H, W) in [0, 1]. NO flips (laterality unknown).
        # DTYPE SAFETY: rng.rand()/randn() return float64; cast back to float32
        # or the GPU forward crashes (Double input vs Float weights).
        if self.rng.rand() < 0.5:  # brightness/contrast jitter
            b = np.float32(0.9 + 0.2 * self.rng.rand())
            c = np.float32(0.9 + 0.2 * self.rng.rand())
            img = np.clip((img - 0.5) * c + 0.5 * b, 0, 1).astype(np.float32)
        if self.rng.rand() < 0.3:  # gaussian noise
            img = np.clip(img + self.rng.randn(*img.shape).astype(np.float32) * np.float32(0.02), 0, 1).astype(np.float32)
        return img.astype(np.float32)

    def _rotate_vol(self, vol):
        # vol: (S, H, W) float32. Small rotation (±10°) via affine grid.
        # Rotation is laterality-safe (no mirroring).
        # DTYPE SAFETY: np.cos/sin return float64; grid_sample is strict.
        # Force everything to float32 explicitly.
        ang = float(self.rng.uniform(-10, 10)) * float(np.pi) / 180.0
        ca, sa = float(np.cos(ang)), float(np.sin(ang))
        theta = torch.tensor([[[ca, -sa, 0.0], [sa, ca, 0.0]]],
                             dtype=torch.float32)
        t = torch.from_numpy(vol).unsqueeze(1).float()  # (S,1,H,W) float32
        grid = F.affine_grid(theta.expand(t.size(0), -1, -1), t.size(),
                             align_corners=False).to(t.dtype)
        t = F.grid_sample(t, grid, mode='bilinear', padding_mode='zeros',
                          align_corners=False)
        return t.squeeze(1).numpy()

    def __getitem__(self, i):
        uid = self.uids[i]
        planes = {}
        mask = []
        have = set(self.store.planes_of(uid))
        for p in PLANES:
            if p in have:
                vol = self.store.get_plane(uid, p)  # (S, 336, 336) uint8
                vol = sample_slices(vol, self.n_slices).astype(np.float32) / 255.0
                if self.augment:
                    vol = np.stack([self._augment(s) for s in vol])
                    if self.rng.rand() < 0.5:
                        vol = self._rotate_vol(vol)
                # resize to IMAGE_SIZE (bilinear) via torch
                t = torch.from_numpy(vol).unsqueeze(1)  # (S,1,H,W)
                t = F.interpolate(t, size=(self.image_size, self.image_size),
                                  mode='bilinear', align_corners=False)
                vol = t.squeeze(1).numpy()
                planes[p] = vol
                mask.append(1.0)
            else:
                planes[p] = np.zeros((self.n_slices, self.image_size,
                                      self.image_size), dtype=np.float32)
                mask.append(0.0)
        targets, loss_mask, sample_w, is_gold = self.label_pack[uid]
        # gray -> 3ch (force float32: augmentation must not leak float64)
        planes3 = {p: np.repeat(v[:, None, :, :], 3, axis=1).astype(np.float32) for p, v in planes.items()}
        return {
            'uid': uid,
            'planes': {p: torch.from_numpy(v) for p, v in planes3.items()},
            'plane_mask': torch.tensor(mask, dtype=torch.float32),
            'targets': torch.from_numpy(targets),
            'loss_mask': torch.from_numpy(loss_mask),
            'weight': torch.tensor(sample_w, dtype=torch.float32),
            'is_gold': torch.tensor(is_gold),
        }


# dataset self-test: shapes, masks, missing-plane tolerance
_ds = KneeDataset(train_uids[:4], store, label_pack, augment=True)
_s = _ds[0]
print('sample keys:', sorted(_s.keys()))
for p in PLANES:
    print(f'  {p}: {tuple(_s["planes"][p].shape)}')  # expect (16, 3, 256, 256)
print(f'  plane_mask: {_s["plane_mask"].tolist()}')
print(f'  targets: {_s["targets"].shape} | loss_mask sum: {_s["loss_mask"].sum().item():.0f}/12')
assert _s['planes']['axial'].shape == (N_SLICES, 3, IMAGE_SIZE, IMAGE_SIZE)
# find a missing-plane study and verify zeros
_miss = [u for u in train_uids if len(store.planes_of(u)) < 3]
if _miss:
    _d2 = KneeDataset(_miss[:1], store, label_pack)
    _s2 = _d2[0]
    pm = _s2['plane_mask']
    for p, m in zip(PLANES, pm.tolist()):
        if m == 0:
            assert (_s2['planes'][p] == 0).all(), f'{p} not zeroed'
    print(f'missing-plane study {_miss[0]}: mask {pm.tolist()} -> zeroed OK')
else:
    print('(no missing-plane study in probe subset; mask logic unit-tested locally)')
print('dataset self-test: ALL GREEN')


In [ ]:
# ================= MODEL: CoAtNet 2.5D + per-finding attention MIL =================
import timm

print('coatnet candidates:', timm.list_models('coatnet*'))
BACKBONE_NAME = None
for cand in BACKBONE_CANDIDATES:
    if cand in timm.list_models('coatnet*'):
        BACKBONE_NAME = cand
        break
assert BACKBONE_NAME, 'FATAL: no coatnet model in this timm version'
print(f'backbone: {BACKBONE_NAME}')


class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim)
        self.U = nn.Linear(feat_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)

    def forward(self, h):  # h: (B, S, D)
        a = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a = F.softmax(a, dim=1)
        return (a * h).sum(dim=1), a.squeeze(-1)  # (B, D), (B, S)


class KneeCoAtNetMIL(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=PRETRAINED):
        super().__init__()
        # FATAL if no pretrained weights: timm raises when pretrained=True
        # but weights don't exist. Random-init training is worthless, so we
        # crash loud here instead of training silently from scratch.
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0)
        print(f'backbone init: pretrained={pretrained} OK')
        D = self.backbone.num_features
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])

    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # Use reshape (not view): DataParallel can hand us non-contiguous
        # tensors, and view() on those causes CUDA misaligned-address errors.
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)  # (B,12,D)

    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B,12,D)
        return torch.cat([self.heads[f](fused[:, f, :])
                          for f in range(N_FINDINGS)], dim=1)  # (B,12)


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


model = KneeCoAtNetMIL()
print(f'params: {count_params(model):.1f}M')
# shape probe on CPU before GPU
model.eval()
with torch.no_grad():
    _pb = {p: torch.randn(1, 4, 3, 224, 224) for p in PLANES}
    _pm = torch.ones(1, 3)
    _o = model(_pb, _pm)
    print(f'cpu shape probe: in (1,4,3,224,224) x3 -> out {tuple(_o.shape)}')
    assert _o.shape == (1, N_FINDINGS)
print('model self-test: ALL GREEN')


In [ ]:
# ================= TRAINING UTILS =================
def make_loader(uids, augment, batch_size=BATCH_SIZE, shuffle=True, num_workers=4):
    ds = KneeDataset(uids, store, label_pack, augment=augment)

    def collate(batch):
        out = {'uid': [b['uid'] for b in batch]}
        out['planes'] = {p: torch.stack([b['planes'][p] for b in batch]) for p in PLANES}
        for k in ['plane_mask', 'targets', 'loss_mask', 'weight', 'is_gold']:
            out[k] = torch.stack([b[k] for b in batch])
        return out

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True, collate_fn=collate)


def masked_asl(logits, targets, loss_mask, sample_weight):
    # (B,12) element losses -> mask abstained -> mean over active -> sample weight
    el = asl_loss(logits, targets)                       # (B,12)
    per_sample = (el * loss_mask).sum(1) / loss_mask.sum(1).clamp(min=1e-6)
    return (per_sample * sample_weight).mean()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, gm = [], [], []
    for b in loader:
        planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
        logits = model(planes, b['plane_mask'].to(DEVICE))
        ys.append(b['targets'].numpy())
        ps.append(torch.sigmoid(logits).float().cpu().numpy())
        gm.append(b['is_gold'].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    gm = np.concatenate(gm).astype(bool)
    yb = (ys >= 0.5).astype(int)  # binarize soft targets for AUC
    macro, per = macro_auc(yb, ps)
    macro_g, per_g = (None, {})
    if gm.sum() >= 4:
        macro_g, per_g = macro_auc(yb[gm], ps[gm])
    return macro, per, macro_g


def save_ckpt(path, model, opt, epoch, stage, best):
    # torch>=2.6 loads with weights_only=True: np.float64 (what macro_auc
    # returns) pickles as numpy.core.multiarray.scalar, which the weights-only
    # unpickler REJECTS -> crash on load_ckpt. float() keeps the payload clean.
    torch.save({'epoch': epoch, 'stage': stage, 'best': float(best),
                'model': model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict(),
                'opt': opt.state_dict()}, path)
    print(f'  saved {path}')


def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location='cpu')
    tgt = model.module if isinstance(model, nn.DataParallel) else model
    tgt.load_state_dict(ck['model'])
    if opt is not None:
        opt.load_state_dict(ck['opt'])
    print(f"  resumed {path} (stage {ck['stage']}, epoch {ck['epoch']}, best {ck['best']})")
    return ck['epoch'], ck['stage'], ck['best']


In [ ]:
# ================= PROBE MODE: full pipeline on tiny subset =================
# Gate: this cell must print ALL GREEN before any full run.
if PROBE:
    print('=== PROBE: 1 train step + 1 eval + ckpt roundtrip ===')
    pm = KneeCoAtNetMIL().to(DEVICE)
    # NOTE: no DataParallel in probe. DataParallel splits batch=2 across 2 GPUs
    # (1 sample each), which triggers CUDA misaligned-address in CoAtNet's
    # kernels. Single-GPU probe is sufficient as a smoke test.
    popt = torch.optim.AdamW(pm.parameters(), lr=LR_S1, weight_decay=WEIGHT_DECAY)
    ptr = make_loader(train_uids, augment=True, batch_size=2, shuffle=True, num_workers=0)
    pva = make_loader(val_uids, augment=False, batch_size=2, shuffle=False, num_workers=0)
    b = next(iter(ptr))
    print(f"  batch planes: {tuple(b['planes']['axial'].shape)} (expect (2,{N_SLICES},3,{IMAGE_SIZE},{IMAGE_SIZE}))")
    planes = {p: v.to(DEVICE) for p, v in b['planes'].items()}
    t0 = time.time()
    logits = pm(planes, b['plane_mask'].to(DEVICE))
    print(f'  forward: {tuple(logits.shape)} in {time.time()-t0:.1f}s')
    assert logits.shape == (2, N_FINDINGS)
    loss = masked_asl(logits, b['targets'].to(DEVICE), b['loss_mask'].to(DEVICE),
                      b['weight'].to(DEVICE))
    assert torch.isfinite(loss), 'FATAL: NaN loss on probe'
    print(f'  loss: {loss.item():.4f} (finite OK)')
    popt.zero_grad(); loss.backward()
    gn = torch.nn.utils.clip_grad_norm_(pm.parameters(), 1.0)
    popt.step()
    print(f'  backward+step OK (grad norm {gn:.2f})')
    if torch.cuda.is_available():
        print(f'  gpu mem: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB peak')
    macro, per, macro_g = evaluate(pm, pva)
    print(f'  val macro AUC (weak): {macro:.4f}' if macro else '  val macro AUC: n/a')
    save_ckpt(f'{CKPT_DIR}/probe.pt', pm, popt, 0, 'probe', macro or 0)
    load_ckpt(f'{CKPT_DIR}/probe.pt', pm, popt)
    print('PROBE: ALL GREEN — full run cleared')
else:
    print('PROBE=False — skipping probe (only do this after a green probe)')


In [ ]:
# ================= STAGE 1: weak-label training =================
def run_stage(uids_tr, uids_va, epochs, lr, stage_name, start_epoch=0, best=-1):
    model = KneeCoAtNetMIL().to(DEVICE)
    # DataParallel DISABLED: CoAtNet's CUDA kernels crash with misaligned-address
    # under DataParallel on T4s (proven by probe v8/v9/v10 saga). Single-GPU only.
    # BATCH_SIZE=2 keeps per-GPU images at 48 (probe peak 13.76/14.56 GB).
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    if RESUME:
        ck_ep, ck_stage, best = load_ckpt(RESUME, model, opt)
        # Same-stage resume continues after the saved epoch. A stage-1 ckpt
        # used for stage 2 starts stage 2 at epoch 0 -- stage epoch counters
        # are independent (stage-1's epoch 7 would make range(8, EPOCHS_S2)
        # empty and silently skip stage 2 entirely).
        start_epoch = ck_ep + 1 if ck_stage == stage_name else 0
        # load_ckpt restores the saved optimizer state INCLUDING its LR (stage-1's).
        # Enforce this stage's LR explicitly; keep the warm momentum buffers.
        for pg in opt.param_groups:
            pg['lr'] = lr
    tr_loader = make_loader(uids_tr, augment=True)
    va_loader = make_loader(uids_va, augment=False, shuffle=False)
    steps_per_epoch = math.ceil(len(uids_tr) / BATCH_SIZE)
    print(f'{stage_name}: {len(uids_tr)} train / {len(uids_va)} val | '
          f'{steps_per_epoch} steps/ep x {epochs} eps')
    for ep in range(start_epoch, epochs):
        model.train()
        t_ep, tot, n = time.time(), 0.0, 0
        opt.zero_grad()
        for si, b in enumerate(tr_loader):
            planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
            logits = model(planes, b['plane_mask'].to(DEVICE))
            loss = masked_asl(logits, b['targets'].to(DEVICE),
                              b['loss_mask'].to(DEVICE), b['weight'].to(DEVICE))
            if not torch.isfinite(loss):
                raise RuntimeError(f'KILL GATE: NaN/Inf loss at ep{ep} step{si}')
            (loss / ACCUM_STEPS).backward()
            if (si + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            tot += loss.item(); n += 1
        # leftover accum
        if n % ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        sched.step()
        macro, per, macro_g = evaluate(model, va_loader)
        dt = time.time() - t_ep
        print(f'  ep{ep+1}/{epochs} | loss {tot/n:.4f} | '
              f'val macro {macro:.4f}' if macro else f'  ep{ep+1} val n/a',
              f'| gold-macro {macro_g:.4f}' if macro_g else '', f'| {dt:.0f}s',
              flush=True)
        improved = macro is not None and macro > best
        if improved:
            best = macro
            save_ckpt(f'{CKPT_DIR}/{stage_name}_best.pt', model, opt, ep, stage_name, best)
        save_ckpt(f'{CKPT_DIR}/{stage_name}_last.pt', model, opt, ep, stage_name, best)
    return f'{CKPT_DIR}/{stage_name}_best.pt', best


if not PROBE:
    best1_ckpt, best1 = run_stage(train_uids, val_uids, EPOCHS_S1, LR_S1, 'stage1')
    print(f'STAGE 1 done: best val macro AUC {best1:.4f} -> {best1_ckpt}')
else:
    print('(probe mode: stage 1 skipped)')


In [ ]:
# ================= STAGE 2: gold fine-tune (low LR, light aug) =================
# Fine-tune on the 58 gold studies only. Gold-58 is a REGRESSION GUARD:
# it cannot resolve ±0.01 effects (SE ~0.02). Never tune on it.
if not PROBE:
    # Exclude val-fold gold: training on them leaks into the val AUC
    # (and badly inflates gold-macro), corrupting best-checkpoint selection.
    n_val_gold = sum(1 for u in val_uids if label_pack[u][3])
    gold_uids = [u for u in train_uids if label_pack[u][3]]
    print(f'gold studies for stage 2: {len(gold_uids)} train (+{n_val_gold} held in val)')
    assert len(gold_uids) + n_val_gold == 58, \
        f'expected 58 gold total, got {len(gold_uids)} train + {n_val_gold} val'
    # init stage-2 from stage-1 best (in-memory if stage 1 ran, else from disk)
    _s1 = globals().get('best1_ckpt', f'{CKPT_DIR}/stage1_best.pt')
    assert os.path.exists(_s1), f'FATAL: stage-1 ckpt not found at {_s1}'
    RESUME = _s1
    best2_ckpt, best2 = run_stage(gold_uids, val_uids, EPOCHS_S2, LR_S2, 'stage2')
    print(f'STAGE 2 done: best val macro AUC {best2:.4f} -> {best2_ckpt}')
    print(f'\nFINAL ARTIFACTS in {CKPT_DIR}/: stage1_best.pt, stage2_best.pt')
else:
    print('(probe mode: stage 2 skipped)')


In [ ]:
# ================= INFERENCE STUB (for the submission kernel) =================
# NOTE: the hidden test studies must go through the SAME Phase 0 preprocessing
# (same shard layout + manifest) before this stub can run on them. That test
# preprocessing is a separate step; this stub assumes preprocessed volumes.
# Test study list is discovered at runtime, never hardcoded.
@torch.no_grad()
def predict_uids(model, uids, batch_size=8):
    model.eval()
    loader = make_loader(uids, augment=False, batch_size=batch_size, shuffle=False)
    rows = []
    for b in loader:
        planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
        probs = torch.sigmoid(model(planes, b['plane_mask'].to(DEVICE)))
        probs64 = probs.double().cpu().numpy()  # float64 end-to-end (no tie penalty)
        for uid, pr in zip(b['uid'], probs64):
            rows.append([uid] + pr.tolist())
    return pd.DataFrame(rows, columns=['StudyInstanceUID'] + LABELS)


# stub self-test on val subset (uses stage-2 best if present)
if not PROBE:
    _m = KneeCoAtNetMIL().to(DEVICE)
    if NGPU > 1:
        _m = nn.DataParallel(_m)
    _ck = f'{CKPT_DIR}/stage2_best.pt'
    load_ckpt(_ck if os.path.exists(_ck) else f'{CKPT_DIR}/stage1_best.pt', _m)
    _sub = predict_uids(_m, val_uids[:8])
    print(f'inference stub: {_sub.shape} (expect (8, 13))')
    assert _sub.shape == (8, 13) and _sub[LABELS].notna().all().all()
    print(_sub.iloc[0].round(3).to_dict())
    print('inference stub: ALL GREEN')
else:
    print('(probe mode: inference stub skipped)')


## Kaggle setup (for Justin)

1. **Datasets to attach** (Add data → Datasets): `phase0runA`, `phase0runB`, `phase0runC`,
   `rsna-knee-phase1-weak-labels-v1` (+ the competition data `rsna-knee-abnormality-detection` for `train.csv`).
2. **Accelerator:** GPU T4 x2. **Internet:** ON (for timm pretrained weights; turn off only for the final submission kernel).
3. **Run order:**
   - Set `PROBE = True`, Run All. Expect `PROBE: ALL GREEN` in ~3–5 min. Check the printed shapes, loss finite, GPU mem < 13 GB.
   - Set `PROBE = False`, Run All. Stage 1 ≈ 8 epochs, Stage 2 ≈ 5 epochs.
   - Kill gates: NaN loss → kill immediately; val macro flat while train loss falls → overfitting, kill and adjust.
4. **Checkpoints** land in `/kaggle/working/phase2_ckpt/` — download `stage2_best.pt` after the run.
5. **Resume:** set `RESUME = '/kaggle/working/phase2_ckpt/stage1_last.pt'` and re-run (only the stage cells re-execute).
